In [59]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from tensorflow.keras.models import load_model
import joblib
from sklearn.metrics import (accuracy_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, roc_curve, auc)
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [56]:
train_folder = '/content/drive/MyDrive/lhc_dataset/dataset/train'
train_file = []
train_file = [os.path.join(train_folder, f) for f in os.listdir(train_folder) if f.endswith('.h5')]

all_targets = []
for fpath in train_file:
    with h5py.File(fpath, 'r') as f:
        targets = f['target'][:]
        all_targets.append(targets)

combined_targets = np.concatenate(all_targets)

In [57]:
label_encoder = LabelEncoder()
label_encoder.fit(combined_targets)

LabelEncoder()

In [3]:
cnn_model = load_model("/content/drive/My Drive/lhc_dataset/best_model.h5")
tabular_model = joblib.load("/content/drive/My Drive/lhc_dataset/Copy of final_jet_classifier.joblib")

In [70]:
cnn_model.build((None, 100, 100, 1))
cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [ ]:
y_true_all, y_pred_cnn_all, y_pred_tab_all, y_prob_cnn_all, y_prob_tab_all = [],[],[],[],[]
y_prob_tab_all_raw, y_prob_cnn_all_raw = [],[]
val_files = sorted(glob('/content/drive/My Drive/lhc_dataset/dataset/val/*.h5'))

for f_path in val_files:
  with h5py.File(f_path, 'r') as f:
    x_img = f['jetImage'][:]
    x_img = x_img[..., np.newaxis]
    x_img = x_img.astype('float32') / 255.0
    x_tab = f['jets_data'][:, 1:]
    y_true = f['target'][:]
    y_true_int = label_encoder.transform(y_true)  # map to integers
    y_true_all.append(y_true_int)

    y_prob_cnn_list = []
    batch_size = 64
    for i in range(0, x_img.shape[0], batch_size):
        batch = x_img[i:i+batch_size]
        y_prob_cnn_list.append(cnn_model.predict(batch))
    y_prob_cnn = np.concatenate(y_prob_cnn_list, axis=0)

    if y_prob_cnn.ndim > 1 and y_prob_cnn.shape[1] > 1:
        y_pred_cnn = np.argmax(y_prob_cnn, axis=1)
    else:
        y_pred_cnn = (y_prob_cnn > 0.5).astype(int).ravel()
    y_pred_cnn_all.append(y_pred_cnn)
    y_prob_cnn_all.append(y_prob_cnn)

    y_prob_tab = tabular_model.predict_proba(x_tab)
    if y_prob_tab.shape[1] > 1:
        y_pred_tab = np.argmax(y_prob_tab, axis=1)
    else:
        y_pred_tab = (y_prob_tab > 0.5).astype(int).ravel()
    y_pred_tab_all.append(y_pred_tab)
    y_prob_tab_all.append(y_prob_tab)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
2/2 ━━━━━━━

In [63]:
y_prob_cnn_raw = cnn_model.predict(x_img, batch_size = 64, verbose = 1)
if y_prob_cnn_raw.ndim >1 and y_prob_cnn_raw.shape[1]>1:
  y_pred_cnn = np.argmax(y_prob_cnn_raw, axis = 1)
  y_prob_cnn = np.max(y_prob_cnn_raw, axis = 1)
else:
  y_pred_cnn = (y_prob_cnn_raw>0.5).astype(int).ravel()
  y_prob_cnn = y_prob_cnn_raw.ravel()

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step


In [34]:
y_prob_tab_raw = tabular_model.predict_proba(x_tab)
if y_prob_tab_raw.shape[1]>1:
  y_pred_tab = np.argmax(y_prob_tab_raw, axis = 1)
  y_prob_tab = np.max(y_prob_tab_raw, axis = 1)
else:
  y_pred_tab = (y_prob_tab_raw>0.5).astype(int).ravel()
  y_prob_tab = y_prob_tab_raw.ravel()

In [64]:
y_true_all.append(y_true)
y_pred_cnn_all.append(y_pred_cnn)
y_prob_cnn_all.append(y_prob_cnn)
y_prob_cnn_all_raw.append(y_prob_cnn_raw)
y_pred_tab_all.append(y_pred_tab)
y_prob_tab_all.append(y_prob_tab)
y_prob_tab_all_raw.append(y_prob_tab_raw)

y_true_all = np.concatenate(y_true_all)
y_pred_cnn_all = np.concatenate(y_pred_cnn_all)
y_pred_tab_all = np.concatenate(y_pred_tab_all)
y_prob_cnn_all = np.concatenate(y_prob_cnn_all)
y_prob_tab_all = np.concatenate(y_prob_tab_all)
y_prob_cnn_all_raw = np.concatenate(y_prob_cnn_all_raw)
y_prob_tab_all_raw = np.concatenate(y_prob_tab_all_raw)

In [66]:
y_true_all = np.array([x.decode('utf-8') if isinstance(x, bytes) else x for x in y_true_all])

y_pred_cnn_all = np.array([x.decode('utf-8') if isinstance(x, bytes) else x for x in y_pred_cnn_all])
y_pred_tab_all = np.array([x.decode('utf-8') if isinstance(x, bytes) else x for x in y_pred_tab_all])

In [67]:
unique_labels = np.unique(y_true_all)  # e.g., ['j_g', 'j_q', ...]
label_map = {label: i for i, label in enumerate(unique_labels)}

y_true_all_int = np.array([label_map[x] for x in y_true_all])


y_pred_cnn_all_int = np.array(y_pred_cnn_all)
y_pred_tab_all_int = np.array(y_pred_tab_all)

In [69]:
print(x_img.dtype, x_img.shape)

float32 (10000, 100, 100, 1)


In [68]:
avg_t = 'weighted' if len(np.unique(y_true_all))>2 else 'binary'
cnn_metrics = {
    "Accuracy": accuracy_score(y_true_all, y_pred_cnn_all),
    "Precision": precision_score(y_true_all_int, y_pred_cnn_all, average=avg_t, zero_division=0),
    "Recall": recall_score(y_true_all_int, y_pred_cnn_all, average=avg_t, zero_division=0),
    "ROC AUC": roc_auc_score(y_true_all, y_prob_cnn_all_raw, multi_class='ovr' if len(np.unique(y_true_all)) > 2 else 'raise')
}

tab_metrics = {
    "Accuracy": accuracy_score(y_true_all, y_pred_tab_all),
    "Precision": precision_score(y_true_all_int, y_pred_tab_all, average=avg_t, zero_division=0),
    "Recall": recall_score(y_true_all_int, y_pred_tab_all, average=avg_t, zero_division=0),
    "ROC AUC": roc_auc_score(y_true_all, y_prob_tab_all_raw, multi_class='ovr' if len(np.unique(y_true_all)) > 2 else 'raise')
}

ValueError: Found input variables with inconsistent numbers of samples: [280000, 10000]

In [40]:
compare_df = pd.DataFrame([cnn_metrics, tab_metrics], index = ['Image', 'Tabular'])

print(compare_df)

         Accuracy  Precision  Recall   ROC AUC
Image         0.0   0.041168  0.2029  0.500000
Tabular       0.0   0.821884  0.8199  0.960898


In [42]:
fig, axes = plt.subplot((1,2))
sns.heatmap(confusion_matrix(y_true_all, y_pred_cnn_all), annot = True, fmt = 'd', cmap = 'Blues', ax = axes[0])
axes[0].set_title('CNN Confusion Matrix')

sns.heat_map(confusion_matrix(y_true_all, y_pred_tab_all), annot = True, fmt = 'd', cmap = 'Blues', ax = axes[1])
axes[1].set_title('RandomForest Confusion Matrix')

plt.tightlayout()
plt.show()

ValueError: Single argument to subplot must be a three-digit integer, not (1, 2)

<Figure size 640x480 with 0 Axes>

In [ ]:
fpr_cnn, tpr_cnn, _ = roc_curve(y_true_all, y_prob_cnn_all)
fpr_tab, tpr_tab, _ = roc_curve(y_true_all, y_prob_tab_all)

plt.figure(figsize=(7,6))
plt.plot(fpr_cnn, tpr_cnn, label=f'CNN (AUC = {auc(fpr_cnn, tpr_cnn):.3f})')
plt.plot(fpr_tab, tpr_tab, label=f'RandomForest (AUC = {auc(fpr_tab, tpr_tab):.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()